# Jiang Lab @ SISU — content management notebook

Convenience helpers for the operations the (now-removed) web forms used to handle:

- **`add_news(...)`** — append a row to the News sheet of `site_data.xlsx`.
- **`add_graduation(...)`** — flip a member's row in the People sheet from `current` to `alumni`.
- **`list_news()` / `delete_news(date, title_zh)`** — inspect / remove a News row.
- **`build_all()`** — run `fetch_orcid.py --offline` to refresh every JSON from `site_data.xlsx` (no ORCID HTTP).

All writes go to `site_data.xlsx` first (the canonical store). Run `build_all()` after a batch of edits to regenerate `assets/data/*.json`.

In [ ]:
from __future__ import annotations
import re
import subprocess
import sys
from pathlib import Path
from openpyxl import load_workbook

# Locate project root no matter where the notebook is opened from.
_HERE = Path.cwd()
ROOT = _HERE if (_HERE / "site_data.xlsx").exists() else _HERE.parent
if not (ROOT / "site_data.xlsx").exists():
    raise FileNotFoundError(f"site_data.xlsx not found near {_HERE}")

XLSX = ROOT / "site_data.xlsx"
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))
import fetch_orcid as fo  # noqa: E402

print(f"Project root: {ROOT}")
print(f"xlsx       : {XLSX} ({XLSX.stat().st_size} bytes)")

In [ ]:
# ── Sheet helpers ─────────────────────────────────────────────────────────

def _open(sheet: str):
    wb = load_workbook(XLSX)
    if sheet not in wb.sheetnames:
        raise KeyError(f"Sheet '{sheet}' not in xlsx. Found: {wb.sheetnames}")
    return wb, wb[sheet]


def _headers(ws):
    return [(c.value or "") for c in ws[1]]


def _col(ws, name: str) -> int:
    headers = _headers(ws)
    if name not in headers:
        raise KeyError(f"Sheet '{ws.title}' has no column '{name}'. Available: {headers}")
    return headers.index(name) + 1


def _append(ws, fields: dict):
    headers = _headers(ws)
    unknown = [k for k in fields if k not in headers]
    if unknown:
        raise KeyError(f"Unknown column(s) for sheet '{ws.title}': {unknown}")
    ws.append([fields.get(h, "") for h in headers])

## `add_news`

Append a row to the News sheet. Required: `date` (YYYY-MM-DD), `title_zh`, `tag_zh`, `body_zh`. The summary fields are auto-derived from the body if not provided. `show` defaults to `yes`.

In [ ]:
_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def add_news(date: str, title_zh: str, title_en: str, tag_zh: str, tag_en: str,
             body_zh: str, body_en: str | None = None,
             external_link: str | None = None,
             summary_zh: str | None = None,
             summary_en: str | None = None,
             featured: bool = False):
    """Append a news item to site_data.xlsx (News sheet). Does NOT regen JSON — call build_all() after."""
    if not (date and title_zh and tag_zh and body_zh):
        raise ValueError("date, title_zh, tag_zh, body_zh are all required")
    if not _DATE_RE.match(str(date)):
        raise ValueError(f"date must be YYYY-MM-DD, got {date!r}")

    wb, ws = _open("News")
    fields = {
        "date": date,
        "featured": "yes" if featured else "",
        "tag_zh": tag_zh,
        "tag_en": tag_en or "",
        "title_zh": title_zh,
        "title_en": title_en or "",
        "summary_zh": summary_zh or (body_zh[:80] if body_zh else ""),
        "summary_en": summary_en or ((body_en or "")[:120]),
        "body_zh": body_zh,
        "body_en": body_en or "",
        "link": external_link or "",
        "show": "yes",
    }
    _append(ws, fields)
    wb.save(XLSX)
    print(f"✓ Added news ({date}): {title_zh}")
    print("  Run build_all() to regenerate news.json.")

## `add_graduation`

Flip a People row from `current` to `alumni`. Matches by exact `name_zh`. Sets `status=alumni`, `end_date`, `period`, `next_position` (combined zh/en), and fills `now` if empty.

In [ ]:
def add_graduation(name_zh: str, end_date: str, period: str,
                   next_position_zh: str, next_position_en: str | None = None):
    """Promote a current member to alumni. Matches by exact name_zh. Call build_all() to regen JSON."""
    if not (name_zh and end_date and period and next_position_zh):
        raise ValueError("name_zh, end_date, period, next_position_zh are all required")
    if not _DATE_RE.match(str(end_date)):
        raise ValueError(f"end_date must be YYYY-MM-DD, got {end_date!r}")

    wb, ws = _open("People")
    name_col = _col(ws, "name_zh")
    status_col = _col(ws, "status")
    end_col = _col(ws, "end_date")
    next_col = _col(ws, "next_position")
    period_col = _col(ws, "period")
    now_col = _col(ws, "now")

    target = None
    for r in range(2, ws.max_row + 1):
        v = ws.cell(row=r, column=name_col).value
        if v and str(v).strip() == name_zh.strip():
            target = r
            break
    if not target:
        raise LookupError(
            f"No People row found with name_zh={name_zh!r}. "
            f"Open site_data.xlsx and verify the exact Chinese name."
        )

    next_combined = next_position_zh
    if next_position_en:
        next_combined = f"{next_position_zh} / {next_position_en}"

    ws.cell(row=target, column=status_col, value="alumni")
    ws.cell(row=target, column=end_col, value=end_date)
    ws.cell(row=target, column=next_col, value=next_combined)
    ws.cell(row=target, column=period_col, value=period)
    if not ws.cell(row=target, column=now_col).value:
        ws.cell(row=target, column=now_col, value=next_combined)

    wb.save(XLSX)
    print(f"✓ Promoted to alumni: {name_zh} (row {target}, ended {end_date})")
    print("  Run build_all() to regenerate people.json.")

## `add_person`

Append a new member row to the People sheet. Required: `member_id`, `role`, `name_zh`, `name_en`. `role` must be one of `pi` / `postdoc` / `phd` / `master` / `undergrad` / `ra` / `visiting`. The `_zh` bio fields (`education_zh`, `research_areas_zh`) are optional but recommended — they drive the Chinese version of the people page; the English strings act as a fallback when `_zh` is empty.

In [ ]:
_VALID_ROLES = {"pi", "postdoc", "phd", "master", "undergrad", "ra", "visiting"}


def add_person(member_id: str, role: str, name_zh: str, name_en: str,
               cohort_year: str | int | None = None, status: str = "current",
               title_zh: str = "", title_en: str = "",
               affil_zh: str = "", affil_en: str = "",
               bio_zh: str = "", bio_en: str = "",
               education: str = "", education_zh: str = "",
               research_areas: str = "", research_areas_zh: str = "",
               photo: str = "", email: str = "", orcid: str = "",
               scholar: str = "", homepage: str = "",
               show: str = "yes"):
    """Append a new row to the People sheet. Call build_all() afterwards to regen JSON."""
    if not (member_id and role and name_zh and name_en):
        raise ValueError("member_id, role, name_zh, name_en are all required")
    role = role.lower()
    if role not in _VALID_ROLES:
        raise ValueError(f"role must be one of {sorted(_VALID_ROLES)}, got {role!r}")
    wb, ws = _open("People")
    id_col = _col(ws, "id")
    for r in range(2, ws.max_row + 1):
        if (ws.cell(row=r, column=id_col).value or "") == member_id:
            raise ValueError(f"member_id={member_id!r} already exists in People sheet (row {r})")
    fields = {
        "id": member_id, "status": status, "role": role,
        "cohort_year": str(cohort_year) if cohort_year is not None else "",
        "name_zh": name_zh, "name_en": name_en,
        "title_zh": title_zh, "title_en": title_en,
        "affil_zh": affil_zh, "affil_en": affil_en,
        "bio_zh": bio_zh, "bio_en": bio_en,
        "education": education, "education_zh": education_zh,
        "research_areas": research_areas, "research_areas_zh": research_areas_zh,
        "photo": photo,
        "email": email, "orcid": orcid, "scholar": scholar, "homepage": homepage,
        "show": show,
    }
    _append(ws, fields)
    wb.save(XLSX)
    print(f"✓ Added member: {name_zh} ({member_id}, role={role})")
    print("  Run build_all() to regenerate people.json.")

## `list_news` / `delete_news`

In [ ]:
def list_news() -> list[tuple[str, str, str]]:
    """Return [(date, title_zh, show), ...] for every News row. Prints a table too."""
    _, ws = _open("News")
    d_col = _col(ws, "date")
    t_col = _col(ws, "title_zh")
    s_col = _col(ws, "show")
    rows: list[tuple[str, str, str]] = []
    for r in range(2, ws.max_row + 1):
        t = ws.cell(row=r, column=t_col).value
        if not t:
            continue
        rows.append((
            str(ws.cell(row=r, column=d_col).value or ""),
            str(t),
            str(ws.cell(row=r, column=s_col).value or ""),
        ))
    print(f"{'date':<12} {'show':<6} title_zh")
    print("-" * 60)
    for d, t, s in rows:
        print(f"{d:<12} {s:<6} {t}")
    print(f"\n({len(rows)} row(s))")
    return rows


def delete_news(date: str | None, title_zh: str):
    """Delete row(s) where title_zh matches (and date matches, if provided). Call build_all() after."""
    if not title_zh:
        raise ValueError("title_zh is required")
    wb, ws = _open("News")
    d_col = _col(ws, "date")
    t_col = _col(ws, "title_zh")
    targets: list[int] = []
    for r in range(ws.max_row, 1, -1):
        t = ws.cell(row=r, column=t_col).value
        if not t or str(t).strip() != title_zh.strip():
            continue
        if date is not None and str(ws.cell(row=r, column=d_col).value or "") != str(date):
            continue
        targets.append(r)
    if not targets:
        print(f"No matching News row(s) for date={date!r}, title_zh={title_zh!r}")
        return
    for r in targets:
        ws.delete_rows(r, 1)
    wb.save(XLSX)
    print(f"✓ Deleted {len(targets)} row(s). Run build_all() to regenerate news.json.")

## `build_all`

Runs `python scripts/fetch_orcid.py --offline` in a subprocess. `--offline` skips ORCID HTTP and just rebuilds every JSON file from the current `site_data.xlsx`.

In [ ]:
def build_all():
    """Rebuild assets/data/*.json from site_data.xlsx. No ORCID HTTP. Prints script stdout."""
    res = subprocess.run(
        [sys.executable, str(SCRIPTS / "fetch_orcid.py"), "--offline"],
        capture_output=True, text=True, cwd=str(ROOT),
    )
    print(res.stdout)
    if res.returncode != 0:
        print("--- STDERR ---")
        print(res.stderr)
        raise RuntimeError(f"fetch_orcid.py --offline failed (exit {res.returncode})")

## Examples (uncomment and edit before running)

In [ ]:
# list_news()

In [ ]:
# add_news(
#     date="2026-06-15",
#     title_zh="博士招生公告",
#     title_en="PhD admissions open",
#     tag_zh="博士招生",
#     tag_en="Admissions",
#     body_zh="2027 年秋季入学的博士招生现已开放……",
#     body_en="Applications for Fall 2027 PhD admissions are now open …",
#     external_link="https://ilas.shisu.edu.cn/...",
# )
# build_all()

In [ ]:
# add_graduation(
#     name_zh="张三",
#     end_date="2026-06-30",
#     period="2023–2026",
#     next_position_zh="麦吉尔大学 博士研究生",
#     next_position_en="PhD candidate, McGill University",
# )
# build_all()

In [ ]:
# delete_news("2026-06-15", "博士招生公告")
# build_all()